In [1]:
import torch
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer
from read_jsonl import read_jsonl
import pandas as pd
from captum.attr import LayerIntegratedGradients

In [2]:
tokenizer_distil = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

model_distil = DistilBertForSequenceClassification.from_pretrained("./results/distilbert/checkpoint-170")
model_distil.eval()

# Detect MPS (Apple Silicon GPU)
force_cpu = False
device = torch.device("mps" if torch.backends.mps.is_available() and not force_cpu else "cpu")

model_distil.to(device)


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [3]:
eval_df = read_jsonl("DB-bio/combined_val_and_val_sft_anonymized.jsonl")
len(eval_df)

486

In [4]:
def forward(text, logit = None):
    
    inputs = tokenizer_distil(text,return_tensors="pt", truncation=True, padding='max_length', max_length=128)
    inputs = {k : v.to(device) for k, v in inputs.items()}
    logits = model_distil(**inputs).logits
    if logit is not None:
        return torch.softmax(logits, dim=1)[0][logit].item()
    return torch.argmax(logits).item()

In [5]:
forward("hello")

1

In [6]:
def forward_captum(_input_ids, _attention_mask, _model,):
    out = _model(input_ids=_input_ids, attention_mask=_attention_mask).logits
    return out

lig_distilbert = LayerIntegratedGradients(forward_captum, model_distil.distilbert.embeddings)

def compute_attributions(_text):
    _inputs = tokenizer_distil(_text, return_tensors="pt", truncation=True, padding='max_length', max_length=128)
    _inputs.to(device)
    _input_ids = _inputs['input_ids']
    _input_ids.to(device)
    _attention_mask = _inputs['attention_mask']
    _attention_mask.to(device)
    baseline = torch.zeros_like(_input_ids)
    baseline.to(device)
    
    with torch.no_grad():
        _logits = model_distil(**_inputs).logits
        target = torch.argmax(_logits, dim=1).item()
        conf = torch.softmax(_logits, dim=1)[0][target].item()
    
    _attributions, _delta = lig_distilbert.attribute(
        inputs=_input_ids,
        baselines=baseline,
        additional_forward_args=(_attention_mask,model_distil),
        target=target,
        return_convergence_delta=True
    )
    
    scores = _attributions.sum(dim=-1).squeeze(0)
    tokens = tokenizer_distil.convert_ids_to_tokens(_input_ids[0])
    filtered = [(_input_ids[0][i].item(), tokens[i], scores[i]) for i in range(len(tokens)) if tokens[i] not in ["[CLS]", "[SEP]", "[PAD]", '.', ',', "(", ")"]]
    return filtered

In [18]:
df_rm = pd.read_csv("DB-bio/global_analysis_removed_tokens_top_1K.csv")
df_add = pd.read_csv("DB-bio/global_analysis_added_tokens_top_1K.csv")
removed_tokens = list(df_rm["token_id"])
added_tokens = list(df_add["token_id"])

In [19]:
embedding_matrix = model_distil.get_input_embeddings().weight
embedding_matrix.shape

torch.Size([30522, 768])

In [20]:
embeddings_removed = embedding_matrix[removed_tokens]
embeddings_added = embedding_matrix[added_tokens]
embeddings_removed.shape, embeddings_added.shape

(torch.Size([1000, 768]), torch.Size([1000, 768]))

In [21]:
embeddings_removed = torch.nn.functional.normalize(embeddings_removed, dim=1)
embeddings_added = torch.nn.functional.normalize(embeddings_added, dim=1)
similarity_matrix = torch.matmul(embeddings_removed, embeddings_added.T)

In [22]:
most_similar_for_removed = similarity_matrix.argmax(dim=1)
most_similar_for_added = similarity_matrix.argmax(dim=0)

In [23]:
removed_tokens_replacement = {token: added_tokens[most_similar_for_removed[i]] for i,token in enumerate(removed_tokens)}
added_tokens_replacement = {token: removed_tokens[most_similar_for_added[i]] for i,token in enumerate(added_tokens)}

In [24]:
removed_tokens_replacement[removed_tokens[371]]

2885

In [30]:
[pair for pair in removed_tokens_replacement.items()][:5]

[(2002, 2027), (1010, 1999), (2010, 2037), (1998, 1999), (1032, 0)]

In [25]:
def get_candidates(target):
    importance_scores = compute_attributions(target)

    aggregated = {}
    for i,t, s in importance_scores:
        _, prev = aggregated.get(i, (t,s))
        aggregated[i] = (t, (prev + s))
    
    ordered = sorted(aggregated.items(), key=lambda x: x[1][1], reverse=True)
    return [i for i, (t, s) in ordered if s > 0]

In [26]:
def attack3(target):
    original_pred = forward(target)
    removed_token_set = set()
    replaced_tokens = set()

    next_path = [(tokenizer_distil(target)["input_ids"][1:-1], removed_token_set, replaced_tokens, 0, 1)]
    visited = set()
    found = []
    best_depth = 1000000
    step = 0
    
    while len(next_path) > 0:
        if step > 20:
            return min(found, key=lambda x: x[3]) if len(found) > 0 else None
        step += 1
        
        path, removed_token_set, replaced_tokens, depth, score = next_path.pop(0)
        removed_token_set = removed_token_set.copy()
        replaced_tokens = replaced_tokens.copy()

        #print(depth)
        if depth >= best_depth:
            continue
        for important_token in get_candidates(tokenizer_distil.decode(path))[:5]:
            new_path = []
            for id in path:
                if id == important_token:
                    if id in removed_tokens_replacement:
                        new_path.append(removed_tokens_replacement[id])
                        replaced_tokens.add((important_token, removed_tokens_replacement[id]))
                    else:
                        removed_token_set.add(important_token)
                else:
                    new_path.append(id)
                    
            if tuple(new_path) in visited:
                continue
            else:
                visited.add(tuple(new_path))     
                
            new_path_text = tokenizer_distil.decode(new_path)
            score = forward(new_path_text, original_pred)
            #print("Depth=[{}] - score: [{}] - token: [{}]".format(depth, score, important_token))
            
            if score <= 0.5:
                best_depth = depth
                found.append((new_path_text, removed_token_set, replaced_tokens, depth))
                #print("Attack succesful [depth={}]: \n".format(depth), new_path_text)
                continue
            
            if depth + 1 < best_depth:
                next_path.append((new_path, removed_token_set, replaced_tokens, depth + 1, score))
            
        next_path = sorted(next_path, key=lambda x: x[2])
    return None

In [28]:
for text in eval_df["text"].head(1):
    print(attack3(text))

None


In [27]:
success = 0
total = 0
for text in eval_df["text"]:
    success += int(attack3(text) is not None)
    total += 1
    print("Success rate: " + str(success / total))
    print("----------------------------------------------------------------------------------------------------------------------------------------")

Success rate: 1.0
----------------------------------------------------------------------------------------------------------------------------------------
Success rate: 1.0
----------------------------------------------------------------------------------------------------------------------------------------
Success rate: 0.6666666666666666
----------------------------------------------------------------------------------------------------------------------------------------
Success rate: 0.75
----------------------------------------------------------------------------------------------------------------------------------------
Success rate: 0.8
----------------------------------------------------------------------------------------------------------------------------------------
Success rate: 0.8333333333333334
----------------------------------------------------------------------------------------------------------------------------------------
Success rate: 0.8571428571428571
------

KeyboardInterrupt: 

In [28]:
total

35

In [17]:
text = "Stephen J. Gordon (born 4 September 1986) is a chess grandmaster from Oldham, Greater Manchester, England. In September 2004 he took a break from his A-level studies at The Blue Coat School, Oldham to compete in the thirteenth Monarch Assurance Isle of Man International. In 2005, while still a FIDE Master, he finished 6th in the British Championships ahead of a Grandmaster and several International Masters. At the EU Individual Open Chess Championship held at Liverpool in 2006, he led the tournament after eight rounds and finished a very creditable (joint) second, a half point behind winner Nigel Short and level with Luke McShane among others. Probably his best result to date however, was second place in the 2007 British Championship, narrowly losing his share of the lead in the final round. In previous rounds, he defeated both tournament victor Jacob Aagaard and previous champion Jonathan Rowson. By 2008, his rating had reached grandmaster level, although the title itself had not yet been secured. At the British Championship in Liverpool, he almost repeated his performance of the previous year, by taking a share of third place. He was the British under-21 Champion each consecutive year between 2005 and 2008. He became a grandmaster on 1 August 2009. He has been one of the co-presenters of the chess podcast The Full English Breakfast since its inaugural show in October 2010."
forward(text)

0

In [18]:
text = "Person is a chess grandmaster from europe. they took a break from their A-level studies at The Blue Coat School, Oldham to compete in the thirteenth Monarch Assurance Isle of Man International. while still a FIDE Master, they finished in the uk Championships ahead of a Grandmaster and several International Masters. At the EU Individual Open Chess Championship held at Liverpool, they led the tournament after eight rounds and finished a very creditable joint second, a half point behind winner and level with Person among others. Probably their best result to date however, was second place in the uk Championship, narrowly losing their share of the lead in the final round. In previous rounds, they defeated both tournament victor Person and previous champion Person. By 2008, their rating had reached grandmaster level, although the title itself had not yet been secured. At the uk Championship in Liverpool, they almost repeated their performance of the previous year, by taking a share of third place. They was the uk under-21 Champion each consecutive year between. They became a grandmaster. They has been one of the co-presenters of the chess podcast The Full european Breakfast since its inaugural show."
forward(text)

1